# Fine-tune e5-base trên cặp issue Jira (Colab GPU)

Notebook này chỉ **train**. Việc đo làm trên máy local bằng script có sẵn, để số liệu so được với các lần đo trước.

**Trước khi chạy**
1. Menu *Runtime → Change runtime type → T4 GPU*.
2. Chuẩn bị file `data/processed/train_bi_encoder_clean.jsonl` (khoảng 2,4 MB), sinh bởi `pipeline/build_finetune_pairs.py`. File này đã loại toàn bộ issue của `similar_gold_eval` và 40 query pooling.
3. **Dữ liệu sẽ rời máy sang Google Colab.** Văn bản đã được scrub nhưng vẫn còn tên service, mã lỗi, luồng nghiệp vụ. Hãy kiểm tra quy định của đơn vị trước khi tải lên.

**Nguyên tắc:** số epoch cố định, không dùng bộ đo nào để chọn checkpoint (tránh tuning trên eval).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1" datasets accelerate

## 1. Tải file train lên

In [ ]:
from google.colab import files
up = files.upload()          # chọn train_bi_encoder_clean.jsonl
TRAIN_FILE = next(iter(up))
print(TRAIN_FILE)

## 2. Cấu hình

- `VARIANT = "gold_weak"`: 133 cặp gold + 1.598 cặp weak (mặc định).
- `VARIANT = "gold_only"`: chỉ 133 cặp gold, để so xem nhãn weak giúp hay hại.

Tiền tố `query: ` cho cả hai phía, khớp `eval_similar.py` / `build_pooling_set.py` (issue ↔ issue).

In [ ]:
import json, math, random
from collections import Counter
import numpy as np, torch
from datasets import Dataset

VARIANT = "gold_weak"            # hoac "gold_only"
BASE = "intfloat/multilingual-e5-base"
EPOCHS, BATCH, LR, MAX_LEN, SEED = 3, 32, 2e-5, 256, 13

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

rows = [json.loads(l) for l in open(TRAIN_FILE, encoding="utf-8") if l.strip()]
if VARIANT == "gold_only":
    rows = [r for r in rows if r["source"].startswith("gold")]
print(len(rows), "cap duong", Counter(r["source"] for r in rows))

# Chi giu 2 cot van ban: cot thua (weight/source/key) se bi coi la dau vao cua loss.
train_ds = Dataset.from_dict({"anchor": [r["anchor"] for r in rows],
                              "positive": [r["positive"] for r in rows]})

## 3. Train (MultipleNegativesRankingLoss, không lặp issue trong cùng batch)

In [ ]:
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
from sentence_transformers.training_args import BatchSamplers

model = SentenceTransformer(BASE, device="cuda")
model.max_seq_length = MAX_LEN
loss = losses.MultipleNegativesRankingLoss(model)

steps = math.ceil(len(train_ds) / BATCH) * EPOCHS
args = SentenceTransformerTrainingArguments(
    output_dir=f"ckpt-{VARIANT}",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    learning_rate=LR,
    warmup_steps=max(1, int(0.1 * steps)),
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    prompts={"anchor": "query: ", "positive": "query: "},
    seed=SEED,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)
trainer = SentenceTransformerTrainer(model=model, args=args,
                                     train_dataset=train_ds, loss=loss)
trainer.train()

OUT = f"e5-base-jira-{VARIANT}"
model.save(OUT)
json.dump({"base": BASE, "variant": VARIANT, "n_pairs": len(train_ds),
           "epochs": EPOCHS, "batch": BATCH, "lr": LR, "max_len": MAX_LEN,
           "seed": SEED, "prefix": "query: (ca hai phia)",
           "train_file": TRAIN_FILE}, open(f"{OUT}/finetune_meta.json", "w"), indent=1)
print("saved", OUT)

## 4. Tải model về (khoảng 1 GB). Nếu tải trực tiếp chậm, dùng Google Drive ở ô dưới.

In [ ]:
!zip -qr {OUT}.zip {OUT}
files.download(f"{OUT}.zip")

In [ ]:
# Tuy chon: luu sang Google Drive thay vi tai truc tiep
# from google.colab import drive
# drive.mount("/content/drive")
# !cp {OUT}.zip /content/drive/MyDrive/

## 5. Đo trên máy local

Giải nén vào `D:/viettel/semantic/data/models/e5-base-jira-gold_weak/`, rồi báo lại để chạy:
iettel\semantic\data\models\e5-base-jira-gold_weak\`, rồi báo lại để chạy:
- paraphrase trên **3.175 query sạch** (`--keys-file data/processed/finetune_split_clean.json`), cho cả model gốc và model fine-tune;
- gold `similar_gold_eval.jsonl` (134 query).

Pooling **chưa** đo được ngay: model mới sẽ tìm ra tài liệu chưa có nhãn, phải bổ sung vào pool và chấm trước.